In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/annotated/checkpoints/pre_cell_4.pickle

trying: ['lineitem']


me:  1
trying: ['tpch_parent']
me:  0
trying: ['load_supplier']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['supplier']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_nation']
me:  4
trying: ['load_orders']
me:  2
trying: ['orders']


me:  2
trying: ['nation']
me:  4
trying: ['pd']
me:  0


Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_nation
Declaring variable nation


In [4]:
%%RecordEvent
%%time
### cell 4 ###

# Filter lineitems where receipt date > commit date
mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
lineitem_date = lineitem[mask_date][['L_ORDERKEY', 'L_SUPPKEY']]

# Compute unique supplier counts per order before and after the date filter
o_orig = (
    lineitem[['L_ORDERKEY', 'L_SUPPKEY']]
    .drop_duplicates()
    .groupby('L_ORDERKEY', as_index=False, sort=False)['L_SUPPKEY']
    .nunique()
    .rename(columns={'L_SUPPKEY': 'orig_count'})
)
o_after = (
    lineitem_date.drop_duplicates()
    .groupby('L_ORDERKEY', as_index=False, sort=False)['L_SUPPKEY']
    .nunique()
    .rename(columns={'L_SUPPKEY': 'after_count'})
)

# Identify orders with >1 supplier originally and exactly 1 after the date filter
valid_orders = (
    o_orig[o_orig.orig_count > 1]
    .merge(o_after[o_after.after_count == 1], on='L_ORDERKEY', how='inner')[['L_ORDERKEY']]
)

# Further restrict to orders with status 'F'
orders_f = orders[orders.O_ORDERSTATUS == 'F'][['O_ORDERKEY']]
valid_orders = (
    valid_orders
    .merge(orders_f, left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')[['L_ORDERKEY']]
)

# Filter the date-qualified lineitems to only those valid orders
li_final = lineitem_date[lineitem_date.L_ORDERKEY.isin(valid_orders.L_ORDERKEY)][['L_SUPPKEY']]

# Join with supplier and nation, then aggregate and sort
supplier_sel = supplier[['S_SUPPKEY', 'S_NATIONKEY', 'S_NAME']]
nation_sa = nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']]

total = (
    li_final
    .merge(supplier_sel, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')[['S_NATIONKEY', 'S_NAME']]
    .merge(nation_sa, left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner')[['S_NAME']]
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size': 'NUMWAIT'})
    .sort_values(by=['NUMWAIT', 'S_NAME'], ascending=[False, True])
)

CPU times: user 165 ms, sys: 98.2 ms, total: 264 ms
Wall time: 277 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q21/rewritten/o4_mini_high/checkpoints/post_cell_4_try_3.pickle

migration speed (bps): 789173975.5662642
---------------------------
variables to migrate:
li_final 48
load_lineitem 144
lineitem 48
STORAGE_OPTS 64
supplier_sel 48
tpch_parent 54
nation_sa 48
pd 72
load_supplier 144
DATA_ROOT 80
supplier 48
mask_date 48
valid_orders 48
nation 48
total 48
load_nation 144
o_orig 48
load_orders 144
orders 48
orders_f 48
o_after 48
lineitem_date 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q21/rewritten/o4_mini_high/checkpoints/post_cell_4_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['orders', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['orders', 'lineitem', 'supplier']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['total', 'nation_sa', 'supplier_sel', 'li_final', 'lineitem_date', 'orders_f', 'valid_orders', 'o_after', 'o_orig', 'mask_date']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q21/opt_cell_exec_info_4_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[4], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/annotated/checkpoints/post_cell_4.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['lineitem']


me:  1
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['load_supplier']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['supplier']
me:  3
trying: ['orders_filtered']
me:  5
trying: ['orig_output']
me:  6
trying: ['supplier_filtered']
me:  5
trying: ['total']
me:  5
trying: ['nation']
me:  4
trying: ['load_nation']
me:  4
trying: ['nation_filtered']
me:  5
trying: ['lineitem_filtered']
me:  5
trying: ['lineitem_orderkeys']
me:  5
trying: ['load_orders']
me:  2
trying: ['orders']


me:  2
trying: ['load_lineitem']
me:  1


Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_supplier
Declaring variable supplier
Declaring variable nation
Declaring variable load_nation
Declaring variable orders_filtered
Declaring variable supplier_filtered
Declaring variable total
Declaring variable nation_filtered
Declaring variable lineitem_filtered
Declaring variable lineitem_orderkeys
Declaring variable orig_output
